# Return State Discovery 03 — Joint price–volatility coupling states

**Preregistered historical exploration, not prospective confirmation or a trading strategy.**

We fit a Student-t Bayesian nonparametric HDP-HSMM to the **bivariate** nonoverlapping 30-minute observation `(Z, U)`, where `Z` is signed net return standardized by VD18's **net-return variance** forecast and `U` is realized-variance surprise relative to a **separately TRAIN-fitted HAR log-RV** forecast. This is **not** the univariate product Z×U, because that conflates opposite sign/volatility combinations.

Anonymous states are characterized by bivariate means/covariances and explicit semi-Markov durations. Forecast filtering uses **only completed blocks before the next target**. The primary claim requires better *future raw-return* likelihood than both continuous price–volatility interaction and unchanged AM-T-HAR-Blend. Joint-feature temporal likelihood is a distinct descriptive measure, never directly comparable to raw-return NLL.

Protocol: [`RSD 003`](../../discussions/Return_State_Discovery/003_chatgpt_joint_price_volatility_hsmm_preregistration.md). Already inspected 150-day original VD18 TEST is split within itself; all inferential results are HISTORICAL_EXPLORATORY. `Kcap=16` is a numerical weak-limit truncation, not a chosen economic state count.

## 0. Environment and isolated regression checks

In [2]:
import os,sys,json,time,subprocess
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir(): ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Launch Jupyter inside cloned repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14
from data_engineering.modeling import markov_har_variance as v18
from data_engineering.modeling import hdp_hsmm_return_states as hsmm
from data_engineering.modeling import hdp_hsmm_volatility_price as vs
from data_engineering.modeling import hdp_hsmm_price_vol_coupling as m
cfg=m.Registration();assert cfg.kcap==16 and cfg.horizon_min==30
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
assert HIST.is_dir(),f'Missing Binance five-minute spot history: {HIST}'
OUT=ROOT/'notebooks'/'Return_State_Discovery'/'03';OUT.mkdir(parents=True,exist_ok=True)
START=time.monotonic()
# The child pytest process does not inherit notebook sys.path modifications.
pytest_env=os.environ.copy()
pytest_env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+pytest_env.get('PYTHONPATH','')
test=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_hdp_hsmm_price_vol_coupling.py')],
                    capture_output=True,text=True,cwd=str(ROOT),env=pytest_env)
print(test.stdout[-2500:]);print(test.stderr[-650:]);assert test.returncode==0
print('Registration',json.dumps(m.registration(),indent=2))

............                                                             [100%]
12 passed in 3.14s


Registration {
  "version": "RSD03-0.1.0",
  "primary_assets": [
    "BTC",
    "ETH"
  ],
  "secondary_assets": [
    "SOL",
    "XRP"
  ],
  "horizon_min": 30,
  "kcap": 16,
  "dmax": 96,
  "seeds": [
    1801,
    2909
  ],
  "n_iter": 160,
  "burn": 80,
  "thin": 4,
  "posterior_draws_per_chain": 4,
  "boot": 1000,
  "sign_permutations": 100,
  "null_seeds": [
    311,
    719
  ],
  "status": "HISTORICAL_EXPLORATORY"
}


## 1. Refit the original VD18 pre-TEST 30m return-density baseline and the independent HAR log-RV scale

**Two different variance targets.** `pred_var` from AM-T-HAR-Blend describes net terminal return variance. The independently TRAIN-fitted HAR model predicts `exp(E[log RV30 | past])`, after explicitly undoing its later train-only mapping to terminal return variance. `pred_rv_scale` is the latter conditional log-RV scale; it is not claimed to be unbiased E[RV30].

All model fits below use only the original VD18 TRAIN or VALID periods; no fitted VD18 artifacts from a previous run are imported. Historical TEST used only to construct eligible nonoverlapping future blocks. Each block's realized RV is observed only at its end.

In [3]:
assets=cfg.primary_assets
raw,excluded=v14.load_returns(HIST,tuple(assets))
panel=v14.panelize(raw);del raw
features=v14.make_features(panel)
frame=v14.make_targets(panel,features,v18.HORIZONS)
frame=v18.add_har_lags(frame)
assert set(frame.underlying.unique())==set(assets),(frame.underlying.unique(),excluded)
base,baseline_contract=hsmm.build_historical_30m_blocks(frame,particles=256,cal_limit=512)
base=vs.attach_realized_variance(frame,base)
tr30,va30,te30=v18.complete(frame,30)
family=baseline_contract['har_family']
har_fit=v18.fit_har(tr30,30,v18.FEATURE_SETS[family])
te30=te30.loc[te30.state_time.dt.minute.isin((0,30))].copy()
te30['pred_rv_scale']=m.har_log_rv_scale(te30,har_fit)
assert te30[['underlying','state_time']].duplicated().sum()==0
base=base.merge(te30[['underlying','state_time','pred_rv_scale']],on=['underlying','state_time'],how='left',validate='one_to_one')
assert base.pred_rv_scale.notna().all()
assert base[['R_30','rv30','pred_var','pred_rv_scale','baseline_nll']].notna().all().all()
print('AM-T-HAR-Blend (same pre-TEST fit protocol as RSD01/02):',baseline_contract)
print('TRAIN-only HAR log RV family:',family,'| excluded:',excluded)
print(base.groupby('underlying').agg(blocks=('R_30','size'),rv0=('rv30',lambda x:int((x==0).sum()))))

AM-T-HAR-Blend (same pre-TEST fit protocol as RSD01/02): {'har_family': 'HAR_4', 'blend_weight': 0.8395717427385103, 'base_scale': 1.4756492057728403, 'base_shape': 1.826083232241286, 'historical_test_origin_count': 14398}
TRAIN-only HAR log RV family: HAR_4 | excluded: {}
            blocks  rv0
underlying             
BTC           7199    0
ETH           7199    0


## 2. Construct the joint price–volatility feature with strict chronology

No overlap: one origin at UTC :00 / :30, followed by six observed five-minute returns. The joint observation is available after 30 minutes, not at its origin. Train-only RV floors and robust normalizers are frozen before evaluation. The middle 20% of old VD18 TEST is a diagnostic partition; the final 20% is a historical exploratory evaluation.

In [4]:
days=sorted(pd.to_datetime(base.state_time,utc=True).dt.floor('D').unique())
assert len(days)>=120,len(days)
cut1,cut2=days[int(len(days)*.6)],days[int(len(days)*.8)]
base['period']=np.where(base.state_time<cut1,'fit',np.where(base.state_time<cut2,'diagnostic','evaluation'))
print('Chronological cutoffs:',cut1,cut2,'| old VD18 TEST days',len(days))
blocks=[];floors={}
for asset,g in base.groupby('underlying',sort=True):
    positive=g.loc[(g.period=='fit')&(g.rv30>0),'rv30'].to_numpy(float)
    floor=max(.001*float(np.median(positive)) if len(positive) else 1e-12,1e-16)
    b=m.joint_features(g,rv_floor=floor)
    b['underlying']=asset;b['period']=g.sort_values('state_time').period.to_numpy()
    blocks.append(b);floors[asset]=floor
joint=pd.concat(blocks,ignore_index=True)
assert np.isfinite(joint[['joint_1','joint_2','coupling']].to_numpy()).all()
print('Train-only RV floor',floors)
print(joint.groupby(['underlying','period']).agg(n=('R_30','size'),z_mean=('joint_1','mean'),u_mean=('joint_2','mean')))

def contiguous(g,min_len):
    q=g.sort_values('state_time').reset_index(drop=True)
    segments=hsmm.contiguous_segments(q.state_time,min_length=min_len)
    assert segments,f'No contiguous segments with >={min_len} blocks'
    selected=np.concatenate([np.arange(a,b) for a,b in segments])
    clean=q.iloc[selected].reset_index(drop=True)
    bounds=np.r_[0,np.cumsum([b-a for a,b in segments])].astype(int)
    return clean,bounds

data={}
for asset in assets:
    group=joint[joint.underlying==asset]
    train,tb=contiguous(group[group.period=='fit'],min_len=24)
    ev,eb=contiguous(group[group.period=='evaluation'],min_len=1)
    diag,db=contiguous(group[group.period=='diagnostic'],min_len=1)
    norm=m.fit_joint_normalizer(train)
    data[asset]={'train':train,'eval':ev,'diag':diag,'train_bounds':tb,'eval_bounds':eb,'norm':norm}
    print(asset,'fit blocks',len(train),'test blocks',len(ev),'fit segments',len(tb)-1,
          'test segments',len(eb)-1,'train logV norm',norm['logv_center'],norm['logv_scale'])

Chronological cutoffs: 2026-07-21 00:00:00+00:00 2026-08-20 00:00:00+00:00 | old VD18 TEST days 150
Train-only RV floor {'BTC': 3.8445727259596426e-09, 'ETH': 6.042432556461763e-09}
                          n    z_mean    u_mean
underlying period                              
BTC        diagnostic  1440  0.020082 -0.191609
           evaluation  1439  0.027380 -0.068800
           fit         4320 -0.011175 -0.060284
ETH        diagnostic  1440  0.037689 -0.214467
           evaluation  1439  0.028763 -0.137693
           fit         4320 -0.007876 -0.126875
BTC fit blocks 4320 test blocks 1439 fit segments 1 test segments 1 train logV norm -12.05691146908471 0.8358305170317806
ETH fit blocks 4320 test blocks 1439 fit segments 1 test segments 1 train logV norm -11.52457318289723 0.792954517031111


## 3. Student-t HDP-HSMM fit on bivariate `(Z,U)`, unknown occupied state count

Full covariance matrices allow state-dependent price–volatility correlations, but the emissions are Student-t(6) throughout. `Kcap=16` is a finite weak-limit approximation; occupancy under this ceiling is not itself evidence of discrete regimes. Review both chain histograms and null-state counts before interpreting any duration or state profile.

In [5]:
posterior={};chain_rows=[];state_profiles=[]
sp=hsmm.Spec(kcap=cfg.kcap,dmax=cfg.dmax,n_iter=cfg.n_iter,burn=cfg.burn,thin=cfg.thin,seed=cfg.seeds[0])
for asset in assets:
    d=data[asset]
    train=d['train'];norm=d['norm']
    X=hsmm.apply_normalizer(train[['joint_1','joint_2']].to_numpy(float),norm)
    chains=[]
    for ci,seed in enumerate(cfg.seeds,1):
        fit=hsmm.fit_posterior(X,hsmm.Spec(**{**sp.__dict__,'seed':seed}),segments=d['train_bounds'])
        summary=hsmm.posterior_summary(fit)
        chain_rows.append({'asset':asset,'chain':ci,**{k:v for k,v in summary.items() if k!='K_histogram'},
                           'K_histogram':json.dumps(summary['K_histogram'])})
        q=m.posterior_coupling_state_profiles(fit,chain=ci);q['asset']=asset
        state_profiles.append(q)
        print(asset,'chain',ci,'occupied K:',summary['K_histogram'],'MH:',summary['beta_mh_acceptance'],
              'warning:',summary['warning'])
        chains.append(fit)
    posterior[asset]=chains
chains_table=pd.DataFrame(chain_rows)
states_table=pd.concat(state_profiles,ignore_index=True)
display(chains_table)
display(states_table[['asset','chain','state_id_in_draw','train_occupancy',
                       'emission_z_mean_normalized','emission_u_mean_normalized',
                       'emission_z_u_correlation','median_episode_30m_blocks']].round(3))

BTC chain 1 occupied K: {'6': 19, '7': 1} MH: 0.71875 warning: NONE
BTC chain 2 occupied K: {'6': 13, '7': 7} MH: 0.58125 warning: NONE
ETH chain 1 occupied K: {'5': 15, '6': 4, '7': 1} MH: 0.6875 warning: NONE
ETH chain 2 occupied K: {'7': 11, '8': 9} MH: 0.6125 warning: NONE


,asset,chain,n_draws,K_occupied_median,K_occupied_05_95,at_Kcap_probability,at_duration_cap_mean,beta_mh_acceptance,warning,K_histogram
0,BTC,1,20,6.0,"[6.0, 6.050000000000001]",0.0,0.0,0.71875,NONE,"{""6"": 19, ""7"": 1}"
1,BTC,2,20,6.0,"[6.0, 7.0]",0.0,0.0,0.58125,NONE,"{""6"": 13, ""7"": 7}"
2,ETH,1,20,5.0,"[5.0, 6.050000000000001]",0.0,0.0,0.68750,NONE,"{""5"": 15, ""6"": 4, ""7"": 1}"
3,ETH,2,20,7.0,"[7.0, 8.0]",0.0,0.0,0.61250,NONE,"{""7"": 11, ""8"": 9}"


,asset,chain,state_id_in_draw,train_occupancy,emission_z_mean_normalized,emission_u_mean_normalized,emission_z_u_correlation,median_episode_30m_blocks
0,BTC,1,0,783,1.225,0.574,0.697,1.0
1,BTC,1,2,1160,-0.701,-0.121,-0.391,1.0
2,BTC,1,3,938,0.542,-0.341,0.369,1.0
3,BTC,1,5,603,-1.176,0.838,-0.666,1.0
4,BTC,1,8,51,0.168,-0.888,0.017,3.0
5,BTC,1,15,785,0.051,-0.790,-0.031,1.0
6,BTC,2,1,438,-0.622,-0.659,-0.509,1.0
7,BTC,2,2,984,1.052,0.490,0.653,1.0
8,BTC,2,4,662,-0.133,-0.367,-0.163,1.0
9,BTC,2,6,1210,-0.992,0.398,-0.636,1.0


## 4. Strictly causal next-return density + joint-feature sequence likelihood

**Primary forecast:** predict the block's signed net return using state weights filtered through *earlier* blocks, before observing this block's Z or U. State-response parameters fitted only on FIT period and frozen. Controls: AM-T-HAR-Blend, continuous `previous Z × previous U` Student-t, recently trained Student-t, state scale-only and *order-ablated* state mixture. All return NLLs are transformed to raw log-return units with exact `log(sqrt(pred_var))` Jacobian.

**Separate temporal ablation:** causal joint HSMM `(Z,U)` NLL against the same emissions with static occupancy weights and a single bivariate Student-t. These scores are in normalized TWO-dimensional feature space and must **not** be compared numerically with signed-return NLL.

In [6]:
all_scores=[];all_boot=[];all_future=[];all_return_profiles=[];evaluation={}
for asset in assets:
    d=data[asset]
    summary,bootstrap,scored,ret_profiles=m.score_price(d['train'],d['eval'],d['norm'],
         d['train_bounds'],d['eval_bounds'],posterior[asset],
         draws_per_chain=cfg.posterior_draws_per_chain,B=cfg.boot)
    all_scores.append({'asset':asset,'eval_blocks':len(scored),'eval_days':scored.date.nunique(),**summary})
    all_boot.append(bootstrap.assign(asset=asset))
    all_return_profiles.extend([{'asset':asset,**v} for v in ret_profiles])
    evaluation[asset]=scored
    for ci,post in enumerate(posterior[asset],1):
        fut=m.volatility_coupling_profiles(d['eval'],d['norm'],post,d['eval_bounds'],chain=ci,min_mass=30)
        fut['asset']=asset;all_future.append(fut)
    print(asset,'evaluated',len(scored),'return NLLs:',scored[[
        'AM_T_HAR_Blend_nll','continuous_coupling_t_nll','HDP_joint_state_nll',
        'recent_t_nll','order_ablated_state_nll']].mean().round(6).to_dict())
scores=pd.DataFrame(all_scores);boot=pd.concat(all_boot,ignore_index=True)
future=pd.concat(all_future,ignore_index=True)
return_profiles=pd.DataFrame(all_return_profiles)
display(scores.T)
display(boot.round(6))
print('Representative state-conditioned NEXT price/volatility outcomes:');display(future.round(4))

BTC evaluated 1439 return NLLs: {'AM_T_HAR_Blend_nll': -4.609073, 'continuous_coupling_t_nll': -4.612965, 'HDP_joint_state_nll': -4.60606, 'recent_t_nll': -4.612568, 'order_ablated_state_nll': -4.605955}
ETH evaluated 1439 return NLLs: {'AM_T_HAR_Blend_nll': -4.337505, 'continuous_coupling_t_nll': -4.335974, 'HDP_joint_state_nll': -4.320366, 'recent_t_nll': -4.333722, 'order_ablated_state_nll': -4.320217}


,0,1
asset,BTC,ETH
eval_blocks,1439,1439
eval_days,30,30
AM_T_HAR_Blend_nll,-4.609073,-4.337505
HDP_joint_state_nll,-4.60606,-4.320366
HDP_scale_only_nll,-4.606103,-4.320418
order_ablated_state_nll,-4.605955,-4.320217
continuous_coupling_t_nll,-4.612965,-4.335974
recent_t_nll,-4.612568,-4.333722
HDP_sign_brier,0.250102,0.250499


,a,b,delta,ci025,ci975,days,asset
0,HDP_joint_state_nll,continuous_coupling_t_nll,0.006907,-0.000435,0.013181,30,BTC
1,HDP_joint_state_nll,AM_T_HAR_Blend_nll,0.002976,-0.006497,0.012355,30,BTC
2,HDP_joint_state_nll,recent_t_nll,0.006506,-0.000073,0.012713,30,BTC
3,HDP_joint_state_nll,HDP_scale_only_nll,0.000042,-0.000168,0.000252,30,BTC
4,HDP_joint_state_nll,order_ablated_state_nll,-0.000106,-0.000402,0.000155,30,BTC
5,HDP_joint_observation_nll,order_ablated_joint_nll,0.000764,-0.011750,0.011493,30,BTC
6,HDP_joint_observation_nll,single_joint_t_nll,-0.145650,-0.172032,-0.122474,30,BTC
7,HDP_sign_brier,continuous_sign_brier,0.000247,-0.000231,0.000693,30,BTC
8,HDP_joint_state_nll,continuous_coupling_t_nll,0.015599,0.007593,0.024347,30,ETH
9,HDP_joint_state_nll,AM_T_HAR_Blend_nll,0.017130,-0.000251,0.034608,30,ETH


Representative state-conditioned NEXT price/volatility outcomes:


,chain,state_id_in_draw,effective_future_blocks,next_mean_z,next_mean_u,next_positive_probability,next_sign_continuation,next_z_u_correlation,next_coupling_mean,next_abs_z,asset
0,1,0,249.6512,0.0357,-0.0671,0.5160,0.4743,0.0482,0.0477,0.7408,BTC
1,1,2,370.6630,0.0118,-0.0538,0.5010,0.4768,0.0595,0.0658,0.7565,BTC
2,1,3,303.6379,0.0225,-0.0407,0.5087,0.4762,0.0453,0.0500,0.7597,BTC
3,1,5,207.8111,0.0428,-0.0727,0.5225,0.4724,0.0145,0.0116,0.7345,BTC
4,1,8,32.1365,0.0157,-0.3469,0.5017,0.4696,0.0959,0.0819,0.6222,BTC
5,1,15,274.9926,0.0359,-0.0862,0.5173,0.4743,0.0424,0.0408,0.7364,BTC
6,2,1,138.5877,0.0011,-0.0428,0.4941,0.4866,0.0695,0.0781,0.7733,BTC
7,2,2,321.6741,0.0247,-0.0625,0.5103,0.4754,0.0490,0.0512,0.7476,BTC
8,2,4,234.9097,0.0478,-0.0591,0.5192,0.4782,0.0540,0.0536,0.7434,BTC
9,2,6,406.5774,0.0319,-0.0502,0.5148,0.4718,0.0366,0.0374,0.7481,BTC


## 5. Two negative controls

- **Regime-free conditional control:** given the actual ex-ante HAR log-RV and AM return-variance paths, simulate six independently drawn five-minute Student-t returns; rebuild both Z and U from the SAME six returns to preserve their mechanical dependence, then run the same HDP-HSMM. This is a conditional no-regime diagnostic, **not** a self-consistent fitted stochastic-volatility/recursive HAR null.
- **Outcome sign permutation:** freeze the already-computed causal predictions; independently flip the sign of evaluation outcomes 100 times. This tests whether the frozen sign probabilities are associated with direction beyond randomization; it does **not** refit the HSMM and is not a full model-selection p-value.

In [7]:
null_rows=[];sign_rows=[]
for asset in assets:
    d=data[asset]
    for seed in cfg.null_seeds:
        null=m.conditional_regime_free_control(d['train'],rv_floor=floors[asset],seed=seed)
        X=hsmm.apply_normalizer(null[['joint_1','joint_2']].to_numpy(float),d['norm'])
        small=hsmm.Spec(**{**sp.__dict__,'seed':seed,'n_iter':100,'burn':60,'thin':4})
        res=hsmm.fit_posterior(X,small,segments=d['train_bounds'])
        report=hsmm.posterior_summary(res)
        null_rows.append({'asset':asset,'null_seed':seed,'null_median_K':report['K_occupied_median'],
            'K_histogram':json.dumps(report['K_histogram']),
            'at_Kcap':report['at_Kcap_probability'],'warning':report['warning']})
    actual=float((evaluation[asset].HDP_sign_brier-evaluation[asset].continuous_sign_brier).mean())
    perms=m.conditional_sign_permutation(evaluation[asset],B=cfg.sign_permutations,seed=2711 if asset=='BTC' else 2713)
    sign_rows.append({'asset':asset,'actual_HDP_minus_cont_Brier':actual,
              'null_5pct':float(np.quantile(perms,.05)),'null_95pct':float(np.quantile(perms,.95)),
              'null_fraction_le_actual':float((perms<=actual).mean()),'n_permutations':len(perms)})
null_table=pd.DataFrame(null_rows);sign_table=pd.DataFrame(sign_rows)
display(null_table);display(sign_table)
print('State counts under conditional no-regime Student-t cannot be read as calibrated p-values.')

,asset,null_seed,null_median_K,K_histogram,at_Kcap,warning
0,BTC,311,5.0,"{""5"": 6, ""6"": 4}",0.0,NONE
1,BTC,719,5.0,"{""5"": 8, ""6"": 2}",0.0,NONE
2,ETH,311,4.0,"{""4"": 9, ""5"": 1}",0.0,NONE
3,ETH,719,6.0,"{""6"": 10}",0.0,NONE


,asset,actual_HDP_minus_cont_Brier,null_5pct,null_95pct,null_fraction_le_actual,n_permutations
0,BTC,0.000248,-0.000859,0.000273,0.94,100
1,ETH,0.001118,-0.000906,0.000579,1.00,100


State counts under conditional no-regime Student-t cannot be read as calibrated p-values.


## 6. Historical exploratory decision; compact reproducibility artifacts

The primary predictive claim requires **negative upper 95% day-bootstrap interval** for BOTH HSMM-minus-continuous and HSMM-minus-original AM-T-HAR on BOTH primary assets. An additional *directional* claim requires sign-Brier improvement and controls. Posterior instability or high truncation occupancy blocks interpreting a state count even if score differences improve. Joint-feature temporal gains alone do not establish a return regime.

In [8]:
check_rows=[]
for asset in assets:
    bs=boot[boot.asset==asset]
    def comp(a,b):return bs.loc[(bs.a==a)&(bs.b==b)].iloc[0]
    a=comp('HDP_joint_state_nll','continuous_coupling_t_nll')
    b=comp('HDP_joint_state_nll','AM_T_HAR_Blend_nll')
    j=comp('HDP_joint_observation_nll','order_ablated_joint_nll')
    q=chains_table[chains_table.asset==asset]
    unstable=bool((q.warning!='NONE').any() or q.K_occupied_median.max()-q.K_occupied_median.min()>1)
    candidate=bool(a.ci975<0 and b.ci975<0 and not unstable)
    check_rows.append({'asset':asset,'HDP_minus_cont_return_NLL':float(a.delta),
          'HDP_minus_cont_upper_CI':float(a.ci975),
          'HDP_minus_VD18_return_NLL':float(b.delta),
          'HDP_minus_VD18_upper_CI':float(b.ci975),
          'HDP_minus_static_JOINT_NLL':float(j.delta),
          'HDP_minus_static_joint_upper_CI':float(j.ci975),
          'chain_or_truncation_warning':unstable,
          'historical_return_candidate':candidate})
checks=pd.DataFrame(check_rows)
verdict=('HISTORICAL_EXPLORATORY_RETURN_CANDIDATE' if checks.historical_return_candidate.all()
         else 'NO_INCREMENTAL_RETURN_STATE')
decision={'status':'HISTORICAL_EXPLORATORY','verdict':verdict,
          'prospective_confirmation':False,'options_trade_approved':False,
          'notes':['Occupied HDP state count is not proof of economic regimes.',
                   'Joint-feature likelihood and signed-return likelihood have DIFFERENT dimensions.',
                   'Historical VD18 test reused; no pristine independent holdout.',
                   'Conditional Student-t null does not recursively refit HAR.',
                   'Prior/convergence sensitivity and genuine prospective testing remain required.']}
display(checks)
print('HISTORICAL EXPLORATORY VERDICT:',verdict)
for filename,df in [('03_posterior_chains.csv',chains_table),('03_state_profiles.csv',states_table),
                     ('03_future_profiles.csv',future),('03_model_scores.csv',scores),
                     ('03_bootstrap.csv',boot),('03_regime_free_null.csv',null_table),
                     ('03_sign_null.csv',sign_table),('03_checks.csv',checks)]:
    df.to_csv(OUT/filename,index=False,float_format='%.10g')
manifest={'module':'Return_State_Discovery_03','version':m.VERSION,'registration':m.registration(),
         'assets':assets,'original_VD18_TEST_days':len(days),
         'hsmm_fit_before':str(cut1),'evaluation_from':str(cut2),
         'baseline':baseline_contract,'har_family_for_logRV':family,
         'har_logRV_scales_training_only':True,'rv_floors_train_only':floors,
         'decision':decision,'checks':checks.to_dict('records'),
         'note':'Ex ante terminal-return variance and HAR log-RV scale are different. Raw returns / posterior paths are NOT saved.'}
(OUT/'03_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
(OUT/'03_decision.json').write_text(json.dumps(decision,indent=2),encoding='utf-8')
print('Saved 10 aggregate artifacts to',OUT,'| elapsed seconds',round(time.monotonic()-START,1))

,asset,HDP_minus_cont_return_NLL,HDP_minus_cont_upper_CI,HDP_minus_VD18_return_NLL,HDP_minus_VD18_upper_CI,HDP_minus_static_JOINT_NLL,HDP_minus_static_joint_upper_CI,chain_or_truncation_warning,historical_return_candidate
0,BTC,0.006907,0.013181,0.002976,0.012355,0.000764,0.011493,False,False
1,ETH,0.015599,0.024347,0.017130,0.034608,0.003579,0.013827,True,False


HISTORICAL EXPLORATORY VERDICT: NO_INCREMENTAL_RETURN_STATE
Saved 10 aggregate artifacts to D:\Projects\deribit_market_data\notebooks\Return_State_Discovery\03 | elapsed seconds 237.2


## Reading the experiment

1. States fitted to `(Z,U)` can be useful as **descriptive distributional components** even when no temporal regimes exist. Conditional Student-t, static-emission mixture, and no-regime controls prevent overclaiming state count.
2. The signed coupling `C=Z*U` is **not** the sole HSMM feature because `C>0` conflates up+vol-expansion with down+vol-compression.
3. Compare **signed-return** NLLs only against signed-return NLLs; compare **joint-feature** NLLs only against joint-feature NLLs. A joint likelihood gain can reflect better modeling of `U` without better future price prediction.
4. `Z` is un-clipped for a valid raw-return density. `U` is clipped only for emission stability; any future extension should check clipping frequency.
5. State-filter probabilities before the block are causal. Any retrospectively smoothed state labeling is only descriptive. Never infer an options trade or regime duration without calibration, prior sensitivity, proper continuous-volatility nulls, and new prospective evidence.